# Predicción de Tiempo de Resolución de Issues de GitHub
**Modelo de IA — Examen Final**

Dado un issue recién creado en GitHub, predice si se resolverá:
- 🟢 **rapido** — menos de 1 día
- 🟡 **normal** — entre 1 y 14 días
- 🔴 **lento** — más de 14 días

**Stack:** Python · scikit-learn · TF-IDF · Random Forest

## 1. Instalación de dependencias

In [ ]:
!pip install pandas numpy scikit-learn scipy

## 2. Importaciones

In [ ]:
import pandas as pd
import numpy as np
import pickle
import warnings
warnings.filterwarnings('ignore')

from scipy.sparse import hstack, csr_matrix
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

print('Dependencias cargadas ✓')

## 3. Carga del dataset

In [ ]:
df = pd.read_csv('data/github_issues_combined.csv')
print(f'Shape: {df.shape}')
print(f'Columnas: {list(df.columns)}')
df.head(3)

In [ ]:
print('Repos con más issues:')
print(df['repo'].value_counts().head(10).to_string())
print(f'\nTotal repos únicos: {df["repo"].nunique()}')

## 4. Cálculo del target — tiempo de resolución

El tiempo se calcula como:
```
resolution_days = (closed_at - created_at).total_seconds() / 86400
```
Luego se convierte a 3 clases: **rapido** (<1d), **normal** (1-14d), **lento** (>14d)

In [ ]:
df['created_at'] = pd.to_datetime(df['created_at'], utc=True, errors='coerce')
df['closed_at']  = pd.to_datetime(df['closed_at'],  utc=True, errors='coerce')
df = df.dropna(subset=['created_at', 'closed_at'])

df['resolution_days'] = (df['closed_at'] - df['created_at']).dt.total_seconds() / 86400
df = df[df['resolution_days'] >= 0]

print('Estadísticas de tiempo de resolución (días):')
print(df['resolution_days'].describe().round(2))

In [ ]:
def clasificar(days):
    if days < 1:    return 'rapido'
    elif days < 14: return 'normal'
    else:           return 'lento'

df['label'] = df['resolution_days'].apply(clasificar)

print('Distribución de clases:')
print(df['label'].value_counts())
print()
print(df['label'].value_counts(normalize=True).round(3))

## 5. Ingeniería de features

El modelo usa dos tipos de features:
- **Texto:** título (×2 para darle más peso) + body → procesado con TF-IDF
- **Numéricas:** 12 columnas estructuradas → normalizadas con StandardScaler

In [ ]:
df['title']  = df['title'].fillna('').astype(str)
df['body']   = df['body'].fillna('').astype(str)
df['labels'] = df['labels'].fillna('').astype(str)

# Título duplicado para darle más peso que el body
df['text_combined'] = df['title'] + ' ' + df['title'] + ' ' + df['body']

# Features derivadas del texto
df['title_word_count'] = df['title'].str.split().str.len()
df['body_word_count']  = df['body'].str.split().str.len()
df['has_code_block']   = df['body'].str.contains('```').astype(int)
df['has_url']          = df['body'].str.contains('http').astype(int)
df['has_question']     = df['title'].str.contains(r'\?').astype(int)
df['label_count']      = df['labels'].apply(lambda x: len([i for i in x.split('|') if i]))
df['title_len']        = df['title'].str.len()

NUM_FEATURES = [
    'has_assignee', 'has_milestone', 'comments', 'day_of_week',
    'hour_of_day', 'title_word_count', 'body_word_count',
    'has_code_block', 'has_url', 'has_question',
    'label_count', 'title_len'
]

print(f'Features numéricas ({len(NUM_FEATURES)}): {NUM_FEATURES}')
print(f'Feature de texto: text_combined')
df[NUM_FEATURES].describe().round(2)

## 6. División del dataset — 80% entrenamiento / 20% test

In [ ]:
X = pd.concat([df['text_combined'].rename('text'), df[NUM_FEATURES]], axis=1)
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,   # semilla fija para resultados reproducibles
    stratify=y         # mantiene la misma proporción de clases en train y test
)

print(f'Train: {len(X_train)} issues ({len(X_train)/len(X)*100:.0f}%)')
print(f'Test:  {len(X_test)} issues ({len(X_test)/len(X)*100:.0f}%)')
print(f'\nDistribución en train:')
print(y_train.value_counts())

## 7. Transformación de features

**TF-IDF** convierte el texto en un vector numérico de 8,000 posiciones.  
**StandardScaler** normaliza las features numéricas a media 0 y desviación 1.  
**hstack()** combina ambas matrices en una sola entrada para el modelo.

In [ ]:
# TF-IDF sobre el texto
tfidf = TfidfVectorizer(
    max_features=8000,     # solo los 8,000 términos más relevantes
    ngram_range=(1, 2),    # unigramas y bigramas ("null pointer", "not working")
    sublinear_tf=True,     # log(TF) para reducir peso de términos muy frecuentes
    stop_words='english',  # elimina: the, is, a, and, etc.
    min_df=3,              # ignora términos que aparecen en menos de 3 issues
)
X_train_tfidf = tfidf.fit_transform(X_train['text'])  # aprende vocabulario Y convierte
X_test_tfidf  = tfidf.transform(X_test['text'])        # solo convierte (sin reaprender)

# StandardScaler sobre numéricas
scaler = StandardScaler()
X_train_num = scaler.fit_transform(X_train[NUM_FEATURES])
X_test_num  = scaler.transform(X_test[NUM_FEATURES])

# Combinar matrices sparse
X_train_final = hstack([X_train_tfidf, csr_matrix(X_train_num)])
X_test_final  = hstack([X_test_tfidf,  csr_matrix(X_test_num)])

print(f'Shape final de entrenamiento: {X_train_final.shape}')
print(f'  → {X_train_tfidf.shape[1]} features TF-IDF + {len(NUM_FEATURES)} numéricas')

## 8. Entrenamiento — Random Forest

In [ ]:
rf = RandomForestClassifier(
    n_estimators=300,        # 300 árboles de decisión independientes
    max_depth=20,            # profundidad máxima de cada árbol
    min_samples_leaf=2,      # mínimo 2 ejemplos por hoja (evita overfitting)
    class_weight='balanced', # compensa el desbalance: lento tiene menos ejemplos
    random_state=42,
    n_jobs=-1,               # usa todos los núcleos del CPU
)

print('Entrenando Random Forest...')
rf.fit(X_train_final, y_train)
print('Entrenamiento completado ✓')

## 9. Evaluación del modelo

In [ ]:
y_pred = rf.predict(X_test_final)

acc = accuracy_score(y_test, y_pred)
f1  = f1_score(y_test, y_pred, average='weighted')
baseline = y_test.value_counts(normalize=True).max()

print(f'Accuracy:              {acc:.4f} ({acc*100:.1f}%)')
print(f'F1 weighted:           {f1:.4f}')
print(f'Baseline (clase mayoría): {baseline:.4f} ({baseline*100:.1f}%)')
print(f'Mejora sobre baseline: +{(acc - baseline)*100:.1f} puntos porcentuales')
print()
print('Reporte por clase:')
print(classification_report(y_test, y_pred))
print('Matriz de confusión (filas=real, columnas=predicho):')
print(confusion_matrix(y_test, y_pred, labels=['rapido', 'normal', 'lento']))

## 10. Features más importantes

In [ ]:
feature_names = tfidf.get_feature_names_out().tolist() + NUM_FEATURES
importances   = rf.feature_importances_
top_idx       = np.argsort(importances)[::-1][:15]

print('Top 15 features más importantes:')
print(f'{"Feature":<25} {"Importancia"}')
print('-' * 40)
for i in top_idx:
    tipo = '(numérica)' if feature_names[i] in NUM_FEATURES else '(TF-IDF)'
    print(f'{feature_names[i]:<25} {importances[i]:.5f}  {tipo}')

## 11. Demo — predecir un issue nuevo

In [ ]:
def predecir_issue(titulo, descripcion='', has_assignee=0, has_milestone=0,
                   comments=0, labels='', day_of_week=1, hour_of_day=10):
    texto = titulo + ' ' + titulo + ' ' + descripcion
    label_count = len([l for l in labels.split('|') if l]) if labels else 0

    X_txt = tfidf.transform([texto])
    X_num_row = scaler.transform([[
        has_assignee, has_milestone, comments, day_of_week, hour_of_day,
        len(titulo.split()), len(descripcion.split()),
        int('```' in descripcion), int('http' in descripcion),
        int('?' in titulo), label_count, len(titulo)
    ]])
    X_final = hstack([X_txt, csr_matrix(X_num_row)])

    pred  = rf.predict(X_final)[0]
    proba = rf.predict_proba(X_final)[0]
    clases = rf.classes_

    iconos  = {'rapido': '🟢', 'normal': '🟡', 'lento': '🔴'}
    tiempos = {'rapido': '< 1 día', 'normal': '1–14 días', 'lento': '> 14 días'}

    print(f'Issue: "{titulo}"')
    print(f'Predicción: {iconos[pred]} {pred.upper()} ({tiempos[pred]})')
    print('Confianza por clase:')
    for c, p in sorted(zip(clases, proba), key=lambda x: -x[1]):
        barra = '█' * int(p * 30)
        print(f'  {iconos[c]} {c:8s} {barra:<30} {p*100:.1f}%')
    print()


predecir_issue(
    titulo='Button click not working on mobile Safari',
    descripcion='On iOS 17, tapping the submit button does nothing. Works on Chrome.',
    has_assignee=1, labels='bug'
)
predecir_issue(
    titulo='Add support for dark mode in dashboard',
    descripcion='It would be great to have a dark mode option in the user settings panel.',
    labels='enhancement'
)
predecir_issue(
    titulo='Refactor authentication module to support OAuth2',
    descripcion='The current auth system only supports basic auth. We need OAuth2 for enterprise clients. This requires changes across multiple services.',
    has_milestone=1, labels='refactor|enhancement'
)

## 12. Guardar modelo entrenado

In [ ]:
artifacts = {
    'tfidf': tfidf,
    'scaler': scaler,
    'model': rf,
    'num_features': NUM_FEATURES,
    'classes': list(rf.classes_),
}
with open('data/model_artifacts_v2.pkl', 'wb') as f:
    pickle.dump(artifacts, f)

print('Modelo guardado en data/model_artifacts_v2.pkl ✓')
print(f'Accuracy final: {acc*100:.1f}%')
print(f'F1 weighted:    {f1:.4f}')